#  HYPERPARAMETER TUNING

In [ ]:

#  CrossValidator on Logistic Regression
# Using LR instead of RF - much faster, less crash risk
# Only 4 combinations x 3 folds = 12 fits but lightweight

print("HYPERPARAMETER TUNING - Logistic Regression")

lr_tuned = LogisticRegression(
    labelCol="arrest",
    featuresCol="features",
    maxIter=10
)

param_grid = ParamGridBuilder() \
    .addGrid(lr_tuned.regParam,        [0.01, 0.1]) \
    .addGrid(lr_tuned.elasticNetParam, [0.0,  0.5]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="arrest",
    metricName="areaUnderROC"
)

cv = CrossValidator(
    estimator=lr_tuned,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    parallelism=2,
    seed=42
)

print("Parameter grid:")
print("  regParam        : [0.01, 0.1]")
print("  elasticNetParam : [0.0, 0.5]")
print("  numFolds        : 3")
print("  parallelism     : 2")
print("\nFitting CrossValidator")

start    = time.time()
cv_model = cv.fit(train_df)
cv_time  = round(time.time() - start, 2)

best_model = cv_model.bestModel
cv_preds   = best_model.transform(test_df)

cv_results = evaluate_model(cv_preds, "LR Tuned (CrossValidator)")
cv_results["train_time_sec"] = cv_time
print(f"\nBest regParam        : {best_model.getRegParam()}")
print(f"Best elasticNetParam : {best_model.getElasticNetParam()}")
print(f"Training time        : {cv_time}s")

save_model(best_model, f"{MODELS_DIR}/lr_tuned_model")

# Save CV scores
cv_scores = pd.DataFrame({
    "avg_auc"   : cv_model.avgMetrics,
    "param_set" : range(len(param_grid))
})
cv_scores.to_csv(
    f"{TABLEAU_DIR}/tableau_cv_scores.csv", index=False
)
print("CV scores saved for Tableau")

HYPERPARAMETER TUNING - Logistic Regression
Parameter grid:
  regParam        : [0.01, 0.1]
  elasticNetParam : [0.0, 0.5]
  numFolds        : 3
  parallelism     : 2

Fitting CrossValidator

Model     : LR Tuned (CrossValidator)
AUC-ROC   : 0.8806
Accuracy  : 0.9187
F1 Score  : 0.9130
Precision : 0.9118
Recall    : 0.9187

Best regParam        : 0.01
Best elasticNetParam : 0.0
Training time        : 654.31s
Model saved to /content/chicago_crimes/models/lr_tuned_model
CV scores saved for Tableau


# SKLEARN BASELINE

In [ ]:
print("SKLEARN BASELINE")

from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.ensemble     import RandomForestClassifier as SklearnRF
from sklearn.metrics      import (
    accuracy_score, f1_score,
    precision_score, recall_score, roc_auc_score
)

# Safe sample size for Colab RAM
SAMPLE_SIZE = 30000
print(f"Using {SAMPLE_SIZE:,} row sample for sklearn")
print("Reason: Sklearn runs on single node - cannot")
print("handle 4.4M rows. Demonstrates PySpark advantage.")

# Safe conversion - collect as pandas first
sample_pd = df_processed \
    .select("features", "arrest") \
    .sample(fraction=0.01, seed=42) \
    .limit(SAMPLE_SIZE) \
    .toPandas()

# Convert sparse vectors safely
import numpy as np

def safe_to_array(v):
    try:
        return v.toArray()
    except:
        return np.zeros(10)

X = np.array([safe_to_array(v) for v in sample_pd["features"]])
y = sample_pd["arrest"].values

split   = int(0.8 * len(X))
X_train = X[:split]
X_test  = X[split:]
y_train = y[:split]
y_test  = y[split:]

print(f"Train : {len(X_train):,} | Test : {len(X_test):,}")

SKLEARN BASELINE
Using 30,000 row sample for sklearn
Reason: Sklearn runs on single node - cannot
handle 4.4M rows. Demonstrates PySpark advantage.
Train : 24,000 | Test : 6,000


In [ ]:
# Free pandas memory immediately
del sample_pd
import gc
gc.collect()

def evaluate_sklearn(model, X_test, y_test, name, train_time):
    preds = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1] \
            if hasattr(model, "predict_proba") else preds

    results = {
        "model"          : name,
        "auc"            : round(roc_auc_score(y_test, proba), 4),
        "accuracy"       : round(accuracy_score(y_test, preds), 4),
        "f1"             : round(f1_score(y_test, preds,
                               average="weighted"), 4),
        "precision"      : round(precision_score(y_test, preds,
                               average="weighted"), 4),
        "recall"         : round(recall_score(y_test, preds,
                               average="weighted"), 4),
        "train_time_sec" : train_time,
        "data_size"      : len(X_train)
    }

    for k, v in results.items():
        print(f"  {k:18s}: {v}")
    return results

# Sklearn LR
print("\nSklearn Logistic Regression")
start      = time.time()
sk_lr      = SklearnLR(max_iter=200, random_state=42)
sk_lr.fit(X_train, y_train)
sk_lr_time = round(time.time() - start, 2)
print(f"Training time : {sk_lr_time}s")
sk_lr_res  = evaluate_sklearn(
    sk_lr, X_test, y_test,
    "Sklearn LR (30k)", sk_lr_time
)

# Free memory before next model
del sk_lr
gc.collect()


Sklearn Logistic Regression
Training time : 12.4s
  model             : Sklearn LR (30k)
  auc               : 0.9025
  accuracy          : 0.8817
  f1                : 0.8744
  precision         : 0.8845
  recall            : 0.8817
  train_time_sec    : 12.4
  data_size         : 24000


27

In [ ]:

# Sklearn RF
print("\nSklearn Random Forest")
start      = time.time()
sk_rf      = SklearnRF(
    n_estimators=30,
    max_depth=8,
    random_state=42,
    n_jobs=1         # single thread
)
sk_rf.fit(X_train, y_train)
sk_rf_time = round(time.time() - start, 2)
print(f"Training time : {sk_rf_time}s")
sk_rf_res  = evaluate_sklearn(
    sk_rf, X_test, y_test,
    "Sklearn RF (30k)", sk_rf_time
)

del sk_rf, X, X_train, X_test
gc.collect()

# PySpark vs Sklearn comparison table
comparison = pd.DataFrame([
    {
        "Framework" : "PySpark MLlib",
        "Model"     : "Logistic Regression",
        "Data Size" : "4.4M rows",
        "AUC"       : lr_results["auc"],
        "Accuracy"  : lr_results["accuracy"],
        "F1"        : lr_results["f1"],
        "Time(s)"   : lr_results["train_time_sec"]
    },
    {
        "Framework" : "PySpark MLlib",
        "Model"     : "Random Forest",
        "Data Size" : "4.4M rows",
        "AUC"       : rf_results["auc"],
        "Accuracy"  : rf_results["accuracy"],
        "F1"        : rf_results["f1"],
        "Time(s)"   : rf_results["train_time_sec"]
    },
    {
        "Framework" : "Sklearn",
        "Model"     : "Logistic Regression",
        "Data Size" : "30k rows",
        "AUC"       : sk_lr_res["auc"],
        "Accuracy"  : sk_lr_res["accuracy"],
        "F1"        : sk_lr_res["f1"],
        "Time(s)"   : sk_lr_res["train_time_sec"]
    },
    {
        "Framework" : "Sklearn",
        "Model"     : "Random Forest",
        "Data Size" : "30k rows",
        "AUC"       : sk_rf_res["auc"],
        "Accuracy"  : sk_rf_res["accuracy"],
        "F1"        : sk_rf_res["f1"],
        "Time(s)"   : sk_rf_res["train_time_sec"]
    }
])

print("\nPySpark vs Sklearn Comparison:")
print(comparison.to_string(index=False))

comparison.to_csv(
    f"{TABLEAU_DIR}/tableau_pyspark_vs_sklearn.csv", index=False
)
print("\nComparison saved for Tableau")


Sklearn Random Forest
Training time : 2.71s
  model             : Sklearn RF (30k)
  auc               : 0.8894
  accuracy          : 0.8267
  f1                : 0.7967
  precision         : 0.8587
  recall            : 0.8267
  train_time_sec    : 2.71
  data_size         : 24000

PySpark vs Sklearn Comparison:
    Framework               Model Data Size    AUC  Accuracy     F1  Time(s)
PySpark MLlib Logistic Regression 4.4M rows 0.8806    0.9181 0.9126    41.80
PySpark MLlib       Random Forest 4.4M rows 0.8400    0.9232 0.9087   951.79
      Sklearn Logistic Regression  30k rows 0.9025    0.8817 0.8744    12.40
      Sklearn       Random Forest  30k rows 0.8894    0.8267 0.7967     2.71

Comparison saved for Tableau
